# legacy_api_wrap

## Example 1: an old positional call still works, with a warning

In [ ]:
# Example 1: legacy_api names the arguments that used to be positional
import warnings
from legacy_api_wrap import legacy_api

@legacy_api("scale", "offset")
def transform(x, *, scale=1.0, offset=0.0):
    return x * scale + offset

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    assert transform(2, 3.0, 1.0) == 7.0  # the old positional form
assert len(caught) == 1 and issubclass(caught[0].category, DeprecationWarning), caught
print("Example 1:", caught[0].message)

## Example 2: the keyword form does not warn

In [ ]:
# Example 2: calls already written the new way are untouched
import warnings
from legacy_api_wrap import legacy_api

@legacy_api("scale")
def transform(x, *, scale=1.0):
    return x * scale

with warnings.catch_warnings():
    warnings.simplefilter("error")
    assert transform(2, scale=5.0) == 10.0
    assert transform(2) == 2.0
print("Example 2: keyword and default calls raise no warning")

## Example 3: a custom warning category

In [ ]:
# Example 3: the category argument picks the warning class (FutureWarning reaches end users by default)
import warnings
from legacy_api_wrap import legacy_api

@legacy_api("copy", category=FutureWarning)
def normalize(values, *, copy=True):
    return [v / sum(values) for v in values] if copy else values

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    assert normalize([1, 3], True) == [0.25, 0.75]
assert caught and caught[0].category is FutureWarning
print("Example 3:", caught[0].category.__name__)

## Example 4: the wrapped signature is the new one

In [ ]:
# Example 4: inspect and functools see the keyword-only signature and the original name and docstring
import inspect
from legacy_api_wrap import legacy_api

@legacy_api("n_comps")
def pca(data, *, n_comps=50):
    """Principal components."""
    return n_comps

params = inspect.signature(pca).parameters
assert params["n_comps"].kind is inspect.Parameter.KEYWORD_ONLY, params
assert pca.__name__ == "pca" and pca.__doc__ == "Principal components."
print("Example 4:", inspect.signature(pca))

## Example 5: too many positional arguments is still an error

In [ ]:
# Example 5: only the declared legacy arguments are accepted positionally
from legacy_api_wrap import legacy_api

@legacy_api("a")
def f(x, *, a=0):
    return x + a

try:
    f(1, 2, 3)
except TypeError as error:
    print("Example 5: refused as expected:", str(error)[:80])
else:
    raise AssertionError("an extra positional argument was accepted")